# IRFA DSCC–QVAR replication (core analyses only)

This notebook installs the pinned core repositories and runs DSCC-λ-GARCH/EGARCH, rolling portfolio evaluation, non-recursive five-variable QVAR(1)/(2), and the HybridMetrics ADF/PP/KPSS battery. It contains no optional NPCQ, TVNCQ or DeepTVAR model code.

**Data rights:** the notebook does not download Yahoo Finance data automatically. Provide two input CSV files that you are legally entitled to use and archive. Do not commit provider data or date-level generated outputs to a public GitHub repository or Zenodo record without permission.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, zipfile, shutil
from google.colab import files

# Set this to your GitHub repository after you create it.
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/IRFA-DSCC-QVAR-Replication.git"
PROJECT_DIR = Path("/content/IRFA-DSCC-QVAR-Replication")

if not PROJECT_DIR.exists():
    if "YOUR_GITHUB_USERNAME" not in GITHUB_REPO_URL:
        subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(PROJECT_DIR)])
    else:
        print("Upload the ZIP of the public GitHub staging folder (or set GITHUB_REPO_URL above).")
        up = files.upload()
        zips = [k for k in up if k.lower().endswith('.zip')]
        if not zips:
            raise FileNotFoundError("Upload a ZIP containing code/replicate_irfa.py and code/build_publication_assets.py")
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall('/content/IRFA-DSCC-QVAR-Replication')

# Accept either public staging layout (code/) or the private author-archive layout (05_Code/).
CODE_DIR = PROJECT_DIR/'code' if (PROJECT_DIR/'code'/'replicate_irfa.py').exists() else PROJECT_DIR/'05_Code'
if not (CODE_DIR/'replicate_irfa.py').exists():
    raise FileNotFoundError(f"Could not find replicate_irfa.py under {CODE_DIR}")
DATA_DIR = PROJECT_DIR/'data' if (PROJECT_DIR/'data').exists() else PROJECT_DIR/'03_Data'/'raw'
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)
print('Code:', CODE_DIR)
print('Data:', DATA_DIR)

## 1. Supply the two licensed input files

Expected filenames in `data/` (public repository layout):

- `price_panel.csv`: `Date` plus `Nasdaq100_ETF`, `Bitcoin`, `Clean_Energy_ETF`, `Oil_Linked_ETF`, `VIX`.
- `returns_decimal.csv`: same columns and exact return dates, decimal log returns (not percent).

The historical author archive used legacy names `verified_adjusted_close_prices.csv` and `verified_log_returns.csv`; the script can read these as fallback names. The original data manifest did not record whether `Close` or adjusted `Close` was used.

In [ ]:
price_candidates = [DATA_DIR/'price_panel.csv', DATA_DIR/'verified_adjusted_close_prices.csv']
return_candidates = [DATA_DIR/'returns_decimal.csv', DATA_DIR/'verified_log_returns.csv']
price_file = next((p for p in price_candidates if p.exists()), None)
return_file = next((p for p in return_candidates if p.exists()), None)

if price_file is None or return_file is None:
    print('Upload price_panel.csv and returns_decimal.csv (or the two legacy-named equivalents).')
    uploaded = files.upload()
    for name in uploaded:
        dest = DATA_DIR/name
        shutil.copy2(name, dest)
    price_file = next((p for p in price_candidates if p.exists()), None)
    return_file = next((p for p in return_candidates if p.exists()), None)
if price_file is None or return_file is None:
    raise FileNotFoundError('Both the price panel and decimal return panel are required.')
print('Price file:', price_file)
print('Return file:', return_file)

## 2. Install pinned core libraries and repositories

QVAR and HybridMetrics are checked out at the commits recorded for the historical run. DSCC is installed from the recorded PyPI version. Installation requires a network-enabled Colab runtime.

In [ ]:
req = CODE_DIR/'requirements.txt'
installer = CODE_DIR/'install_core_repositories.py'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(req)])
subprocess.check_call([sys.executable, str(installer)])
print('Core dependencies installed.')

## 3. Run the complete core model pipeline

The pipeline saves fitted DSCC outputs, rolling portfolio tables, QVAR coefficient matrices and diagnostics, unit-root test results, a run manifest and regenerated publication tables/figures. This is a new run with the reconstructed driver; compare it with the supplied historical output before claiming exact source-level reproduction.

In [ ]:
outdir = Path('/content/IRFA_replication_output')
cmd = [sys.executable, str(CODE_DIR/'replicate_irfa.py'), '--data-dir', str(DATA_DIR), '--outdir', str(outdir)]
subprocess.check_call(cmd)
print('Output folder:', outdir)

## 4. Review the run manifest and generated assets

Check the input hashes, data field disclosure, package versions, repository commits, optimizer convergence, model fit, backtest windows, QVAR settings and the return reconstruction error. The AV/SD statistic is reported as a return-to-volatility ratio rather than the conventional active-return information ratio.

In [ ]:
manifest = json.loads((outdir/'run_manifest.json').read_text())
print(json.dumps(manifest, indent=2))
print('Generated publication tables:')
for p in sorted((outdir/'publication_output'/'tables').glob('*.csv')):
    print(p.name)
print('Generated publication figures:')
for p in sorted((outdir/'publication_output'/'figures').glob('*')):
    print(p.name)

## 5. Package the full generated run for private author archiving

This ZIP contains the generated time-series data, results and date-level outputs. Keep it private unless redistribution rights for every input and derived data file have been confirmed. Do not upload it to public GitHub or Zenodo merely because the code repository is public.

In [ ]:
archive = shutil.make_archive('/content/IRFA_full_generated_core_results', 'zip', outdir)
print('Created private results archive:', archive)
files.download(archive)

## 6. Public GitHub and Zenodo release

Upload only the contents of the public code staging folder. Exclude the input data, `replication_output/`, vendor checkouts and any private credentials. Create a GitHub release, enable Zenodo integration, mint the DOI, then update `CITATION.cff` and the manuscript with the real URL/DOI. The notebook does not contain or request GitHub/Zenodo tokens.